# Bayesian Negative-Binomial Crime Forecasting Experiment

Fits a hierarchical negative-binomial model with an adjacency-based spatial-lag feature and benchmarks it against a per-LSOA historical-mean baseline across several train/test regimes (pre-pandemic, post-pandemic, combined, and an operational forward forecast).

## 1. Setup and imports

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import mean_absolute_error, mean_squared_error

import pymc as pm
import arviz as az

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


## 2. Project paths and package imports

In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "cbl20").exists():
            return candidate
    raise FileNotFoundError(f"Could not find project root containing src/cbl20 from: {start}")

PROJECT_ROOT = find_project_root(Path.cwd())
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cbl20.spatial_adjacency import (
    load_lsoa_centroids,
    validate_lsoa_code_coverage,
    build_knn_adjacency_from_centroids,
    summarize_adjacency,
    add_neighbour_previous_month_feature,
)

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SPATIAL_DIR = DATA_DIR / "spatial"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
TABLES_DIR = OUTPUT_DIR / "tables"

TABLES_DIR.mkdir(parents=True, exist_ok=True)

CRIME_PARQUET_PATH = PROCESSED_DIR / "crimes_filtered_for_model.parquet"
if not CRIME_PARQUET_PATH.exists():
    CRIME_PARQUET_PATH = PROCESSED_DIR / "crimes_clean_dedup_all_years.parquet"

LSOA_CENTROID_CSV = (
    SPATIAL_DIR
    / "Lower_layer_Super_Output_Areas_December_2021_Boundaries_EW_BGC_V5_4163309575471683775.csv"
)

print("Project root:", PROJECT_ROOT)
print("Crime parquet:", CRIME_PARQUET_PATH, CRIME_PARQUET_PATH.exists())
print("LSOA centroid CSV:", LSOA_CENTROID_CSV, LSOA_CENTROID_CSV.exists())


## 3. Experiment configuration

In [ ]:
SELECTED_POLICE_FORCES = [
    "Cheshire Constabulary",
    "Lincolnshire Police",
    "Merseyside Police",
    "Metropolitan Police Service",
    "West Midlands Police",
]

SELECTED_CRIME_TYPES = [
    "Anti-social behaviour",
    "Criminal damage and arson",
    "Drugs",
    "Possession of weapons",
    "Vehicle crime",
]

EXPERIMENTS = [
    {
        "experiment_id": "pre_pandemic",
        "description": "Pre-pandemic regime",
        "train_windows": [("2012-01", "2018-12")],
        "test_window": ("2019-01", "2019-12"),
        "is_operational_forecast": False,
    },
    {
        "experiment_id": "post_pandemic",
        "description": "Post-pandemic regime",
        "train_windows": [("2022-01", "2023-12")],
        "test_window": ("2024-01", "2024-12"),
        "is_operational_forecast": False,
    },
    {
        "experiment_id": "combined_non_pandemic",
        "description": "Combined non-pandemic regime",
        "train_windows": [("2012-01", "2019-12"), ("2022-01", "2024-12")],
        "test_window": ("2025-01", "2025-12"),
        "is_operational_forecast": False,
    },
    {
        "experiment_id": "operational_forecast",
        "description": "Final operational forecasting",
        "train_windows": "all_available",
        "test_window": "next_12_months",
        "is_operational_forecast": True,
    },
]

EXPERIMENTS_TO_RUN = "all"

MAX_LSOAS_PER_MODEL = 100
MIN_TOTAL_CRIMES_PER_LSOA = 1
MIN_CENTROID_COVERAGE = 0.80
ADJACENCY_K_NEIGHBOURS = 5

DRAWS = 300
TUNE = 300
CHAINS = 2
TARGET_ACCEPT = 0.9
RANDOM_SEED = 42

SAVE_RESULTS = True
OUTPUT_TAG = "shared_regime_comparison_bayesian_adj_no_offset"

print("Selected forces:", len(SELECTED_POLICE_FORCES))
print("Selected crime types:", len(SELECTED_CRIME_TYPES))
print("Total force-crime combinations:", len(SELECTED_POLICE_FORCES) * len(SELECTED_CRIME_TYPES))
print("Experiments:", [e["experiment_id"] for e in EXPERIMENTS])


## 4. Load reference data

In [ ]:
lsoa_centroids = load_lsoa_centroids(LSOA_CENTROID_CSV)

data_months = duckdb.sql(f'''
    SELECT
        MIN("Month") AS min_month,
        MAX("Month") AS max_month,
        COUNT(*) AS n_rows
    FROM read_parquet('{CRIME_PARQUET_PATH.as_posix()}')
''').df()

DATA_MIN_MONTH = str(data_months.loc[0, "min_month"])
DATA_MAX_MONTH = str(data_months.loc[0, "max_month"])

print(data_months)
print("Centroid LSOAs:", lsoa_centroids["lsoa_code"].nunique())


## 5. Date and window helpers

In [ ]:
def to_month_start(value: str) -> pd.Timestamp:
    return pd.to_datetime(value, format="%Y-%m")

def month_str(value: pd.Timestamp) -> str:
    return pd.Timestamp(value).strftime("%Y-%m")

def next_month(value: str) -> str:
    return month_str(to_month_start(value) + pd.DateOffset(months=1))

def add_months(value: str, n: int) -> str:
    return month_str(to_month_start(value) + pd.DateOffset(months=n))

def resolve_experiment_windows(experiment: dict) -> dict:
    exp = experiment.copy()
    if exp["train_windows"] == "all_available":
        train_start = DATA_MIN_MONTH
        train_end = DATA_MAX_MONTH
        test_start = next_month(DATA_MAX_MONTH)
        test_end = add_months(test_start, 11)
        exp["train_windows"] = [(train_start, train_end)]
        exp["test_window"] = (test_start, test_end)
    return exp

def windows_to_mask(month_series: pd.Series, windows: list[tuple[str, str]]) -> pd.Series:
    mask = pd.Series(False, index=month_series.index)
    for start, end in windows:
        start_ts = to_month_start(start)
        end_ts = to_month_start(end)
        mask = mask | ((month_series >= start_ts) & (month_series <= end_ts))
    return mask

def experiment_panel_bounds(experiment: dict) -> tuple[str, str]:
    train_windows = experiment["train_windows"]
    test_start, test_end = experiment["test_window"]
    starts = [start for start, _ in train_windows] + [test_start]
    ends = [end for _, end in train_windows] + [test_end]
    return min(starts), max(ends)

def sql_in_list(values: list[str]) -> str:
    return ", ".join("'" + v.replace("'", "''") + "'" for v in values)


## 6. Sample availability check

In [ ]:
def check_selected_sample_availability():
    force_sql = sql_in_list(SELECTED_POLICE_FORCES)
    crime_sql = sql_in_list(SELECTED_CRIME_TYPES)
    out = duckdb.sql(f'''
        SELECT
            "Falls within" AS police_force,
            "Crime type" AS crime_type,
            MIN("Month") AS min_month,
            MAX("Month") AS max_month,
            COUNT(*) AS n_records,
            COUNT(DISTINCT "LSOA code") AS n_lsoas
        FROM read_parquet('{CRIME_PARQUET_PATH.as_posix()}')
        WHERE "Falls within" IN ({force_sql})
          AND "Crime type" IN ({crime_sql})
          AND "LSOA code" IS NOT NULL
        GROUP BY "Falls within", "Crime type"
        ORDER BY "Falls within", "Crime type"
    ''').df()
    return out

availability = check_selected_sample_availability()
display(availability)
print("Expected force-crime rows:", len(SELECTED_POLICE_FORCES) * len(SELECTED_CRIME_TYPES))
print("Available force-crime rows:", len(availability))


## 7. Panel construction and adjacency features

In [ ]:
def build_lsoa_month_panel(
    parquet_path: Path,
    police_force: str,
    crime_type: str,
    panel_start_month: str,
    panel_end_month: str,
    train_windows: list[tuple[str, str]],
    lsoa_reference: pd.DataFrame,
    max_lsoas: int | None = 100,
    min_total_crimes: int = 1,
    min_centroid_coverage: float = 0.80,
) -> tuple[pd.DataFrame, dict]:
    parquet_path_posix = Path(parquet_path).as_posix()

    train_filter = " OR ".join(
        [f'("Month" BETWEEN \'{start}\' AND \'{end}\')' for start, end in train_windows]
    )

    lsoas = duckdb.sql(f'''
        SELECT
            "LSOA code" AS lsoa_code,
            ANY_VALUE("LSOA name") AS lsoa_name
        FROM read_parquet('{parquet_path_posix}')
        WHERE "LSOA code" IS NOT NULL
          AND "Falls within" = '{police_force.replace("'", "''")}'
          AND "Month" BETWEEN '{panel_start_month}' AND '{panel_end_month}'
        GROUP BY "LSOA code"
    ''').df()

    if lsoas.empty:
        raise ValueError(f"No LSOAs found for {police_force} between {panel_start_month} and {panel_end_month}.")

    train_counts_for_lsoa_selection = duckdb.sql(f'''
        SELECT
            "LSOA code" AS lsoa_code,
            COUNT(*) AS total_selected_crimes
        FROM read_parquet('{parquet_path_posix}')
        WHERE "LSOA code" IS NOT NULL
          AND "Falls within" = '{police_force.replace("'", "''")}'
          AND "Crime type" = '{crime_type.replace("'", "''")}'
          AND ({train_filter})
        GROUP BY "LSOA code"
    ''').df()

    if train_counts_for_lsoa_selection.empty:
        raise ValueError(f"No training rows found for {police_force} | {crime_type}.")

    counts = duckdb.sql(f'''
        SELECT
            "LSOA code" AS lsoa_code,
            "Month" AS month,
            COUNT(*) AS crime_count
        FROM read_parquet('{parquet_path_posix}')
        WHERE "LSOA code" IS NOT NULL
          AND "Falls within" = '{police_force.replace("'", "''")}'
          AND "Crime type" = '{crime_type.replace("'", "''")}'
          AND "Month" BETWEEN '{panel_start_month}' AND '{panel_end_month}'
        GROUP BY "LSOA code", "Month"
    ''').df()

    if counts.empty:
        raise ValueError(f"No rows found for {police_force} | {crime_type} in selected panel period.")

    counts["month"] = pd.to_datetime(counts["month"], format="%Y-%m")
    counts["lsoa_code"] = counts["lsoa_code"].astype(str).str.strip()
    lsoas["lsoa_code"] = lsoas["lsoa_code"].astype(str).str.strip()
    train_counts_for_lsoa_selection["lsoa_code"] = train_counts_for_lsoa_selection["lsoa_code"].astype(str).str.strip()

    lsoas = lsoas.merge(train_counts_for_lsoa_selection, on="lsoa_code", how="left")
    lsoas["total_selected_crimes"] = lsoas["total_selected_crimes"].fillna(0)
    lsoas = lsoas[lsoas["total_selected_crimes"] >= min_total_crimes].copy()

    if lsoas.empty:
        raise ValueError(f"No LSOAs had at least {min_total_crimes} training crimes for {crime_type!r}.")

    coverage = validate_lsoa_code_coverage(lsoas["lsoa_code"], lsoa_reference)
    if coverage["coverage_rate"] < min_centroid_coverage:
        raise ValueError(
            f"Low LSOA centroid coverage. Coverage={coverage['coverage_rate']:.3f}; "
            f"matched={coverage['n_matched_lsoas']}; missing={coverage['n_missing_lsoas']}"
        )

    centroid_codes = set(lsoa_reference["lsoa_code"].astype(str))
    n_before = lsoas["lsoa_code"].nunique()
    lsoas = lsoas[lsoas["lsoa_code"].isin(centroid_codes)].copy()
    n_after = lsoas["lsoa_code"].nunique()

    lsoas = lsoas.sort_values("total_selected_crimes", ascending=False)
    if max_lsoas is not None:
        lsoas = lsoas.head(max_lsoas)

    months = pd.date_range(
        start=to_month_start(panel_start_month),
        end=to_month_start(panel_end_month),
        freq="MS",
    )

    panel_index = pd.MultiIndex.from_product(
        [lsoas["lsoa_code"], months],
        names=["lsoa_code", "month"],
    )

    panel = panel_index.to_frame(index=False)
    panel = panel.merge(lsoas[["lsoa_code", "lsoa_name"]], on="lsoa_code", how="left")
    panel = panel.merge(counts, on=["lsoa_code", "month"], how="left")

    panel["crime_count"] = panel["crime_count"].fillna(0).astype(int)
    panel["police_force"] = police_force
    panel["crime_type"] = crime_type
    panel["month_num"] = panel["month"].dt.month
    panel["month_id"] = panel["month_num"] - 1

    min_month = panel["month"].min()
    panel["time_index"] = (
        (panel["month"].dt.year - min_month.year) * 12
        + (panel["month"].dt.month - min_month.month)
    ).astype(float)

    panel["lsoa_id"] = pd.Categorical(panel["lsoa_code"]).codes
    # offset disabled for this experiment (no population scaling)
    panel["log_offset"] = 0.0

    metadata = {
        "n_lsoas_before_centroid_filter": int(n_before),
        "n_lsoas_after_centroid_filter": int(n_after),
        "n_lsoas_dropped_no_centroid": int(n_before - n_after),
        "centroid_coverage": float(coverage["coverage_rate"]),
        "missing_lsoas_sample": coverage["missing_lsoas_sample"],
    }

    return panel.sort_values(["lsoa_code", "month"]).reset_index(drop=True), metadata


def add_adjacency_features_to_panel(
    panel: pd.DataFrame,
    lsoa_reference: pd.DataFrame,
    k_neighbours: int = 5,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    lsoa_codes = sorted(panel["lsoa_code"].dropna().astype(str).unique())

    adjacency = build_knn_adjacency_from_centroids(
        lsoa_reference,
        lsoa_codes=lsoa_codes,
        k=k_neighbours,
    )

    adjacency_summary = summarize_adjacency(adjacency)

    panel_adj = add_neighbour_previous_month_feature(
        panel,
        adjacency,
        output_col="neighbour_prev_month_count",
        scale_output=False,
    )

    return panel_adj, adjacency, adjacency_summary


## 8. Train/test split and feature scaling

In [ ]:
def split_train_test_for_experiment(
    panel: pd.DataFrame,
    experiment: dict,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_mask = windows_to_mask(panel["month"], experiment["train_windows"])
    test_start, test_end = experiment["test_window"]
    test_mask = (panel["month"] >= to_month_start(test_start)) & (panel["month"] <= to_month_start(test_end))

    train = panel[train_mask].copy()
    test = panel[test_mask].copy()

    if train.empty:
        raise ValueError("Training set is empty.")
    if test.empty:
        raise ValueError("Test/forecast set is empty.")

    categories = pd.Categorical(train["lsoa_code"]).categories
    train["lsoa_id"] = pd.Categorical(train["lsoa_code"], categories=categories).codes
    test["lsoa_id"] = pd.Categorical(test["lsoa_code"], categories=categories).codes
    test = test[test["lsoa_id"] >= 0].copy()

    if test.empty:
        raise ValueError("No test rows remain after matching LSOA categories to training set.")

    return train.reset_index(drop=True), test.reset_index(drop=True)


def scale_features_using_train(
    train: pd.DataFrame,
    test: pd.DataFrame,
) -> tuple[pd.DataFrame, pd.DataFrame, dict]:
    train = train.copy()
    test = test.copy()

    time_mean = train["time_index"].mean()
    time_std = train["time_index"].std()
    if time_std == 0 or np.isnan(time_std):
        train["time_scaled"] = 0.0
        test["time_scaled"] = 0.0
        time_std_used = 0.0
    else:
        train["time_scaled"] = (train["time_index"] - time_mean) / time_std
        test["time_scaled"] = (test["time_index"] - time_mean) / time_std
        time_std_used = float(time_std)

    spatial_mean = train["neighbour_prev_month_count"].mean()
    spatial_std = train["neighbour_prev_month_count"].std()
    if spatial_std == 0 or np.isnan(spatial_std):
        train["neighbour_prev_month_scaled"] = 0.0
        test["neighbour_prev_month_scaled"] = 0.0
        spatial_std_used = 0.0
    else:
        train["neighbour_prev_month_scaled"] = (train["neighbour_prev_month_count"] - spatial_mean) / spatial_std
        test["neighbour_prev_month_scaled"] = (test["neighbour_prev_month_count"] - spatial_mean) / spatial_std
        spatial_std_used = float(spatial_std)

    return train, test, {
        "time_mean_train": float(time_mean),
        "time_std_train": time_std_used,
        "spatial_lag_mean_train": float(spatial_mean),
        "spatial_lag_std_train": spatial_std_used,
    }


## 9. Bayesian negative-binomial model

In [ ]:
def fit_bayesian_nb_model(
    train: pd.DataFrame,
    test: pd.DataFrame,
    draws: int = 300,
    tune: int = 300,
    chains: int = 2,
    target_accept: float = 0.9,
    random_seed: int = 42,
) -> tuple[pd.DataFrame, az.InferenceData, pd.DataFrame, dict]:
    coords = {
        "lsoa": np.arange(train["lsoa_id"].nunique()),
        "month_of_year": np.arange(12),
        "obs_id": np.arange(len(train)),
    }

    with pm.Model(coords=coords) as nb_model:
        lsoa_idx = pm.Data("lsoa_idx", train["lsoa_id"].to_numpy(), dims="obs_id")
        month_idx = pm.Data("month_idx", train["month_id"].to_numpy(), dims="obs_id")
        time_scaled = pm.Data("time_scaled", train["time_scaled"].to_numpy(), dims="obs_id")
        spatial_lag = pm.Data("spatial_lag", train["neighbour_prev_month_scaled"].to_numpy(), dims="obs_id")
        log_offset = pm.Data("log_offset", train["log_offset"].to_numpy(), dims="obs_id")

        y = train["crime_count"].to_numpy()

        intercept = pm.Normal("intercept", mu=0.0, sigma=2.0)

        # non-centered parameterisation for the grouped random effects
        sigma_lsoa = pm.HalfNormal("sigma_lsoa", sigma=1.0)
        lsoa_raw = pm.Normal("lsoa_raw", mu=0.0, sigma=1.0, dims="lsoa")
        lsoa_effect = pm.Deterministic("lsoa_effect", lsoa_raw * sigma_lsoa, dims="lsoa")

        sigma_month = pm.HalfNormal("sigma_month", sigma=1.0)
        month_raw = pm.Normal("month_raw", mu=0.0, sigma=1.0, dims="month_of_year")
        month_effect = pm.Deterministic("month_effect", month_raw * sigma_month, dims="month_of_year")

        beta_time = pm.Normal("beta_time", mu=0.0, sigma=1.0)
        beta_spatial_lag = pm.Normal("beta_spatial_lag", mu=0.0, sigma=1.0)

        eta = (
            intercept
            + lsoa_effect[lsoa_idx]
            + month_effect[month_idx]
            + beta_time * time_scaled
            + beta_spatial_lag * spatial_lag
            + log_offset
        )

        mu = pm.math.exp(eta)
        alpha = pm.HalfNormal("alpha", sigma=10.0)

        pm.NegativeBinomial("y_obs", mu=mu, alpha=alpha, observed=y, dims="obs_id")

        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=target_accept,
            random_seed=random_seed,
            progressbar=True,
        )

        pm.set_data(
            {
                "lsoa_idx": test["lsoa_id"].to_numpy(),
                "month_idx": test["month_id"].to_numpy(),
                "time_scaled": test["time_scaled"].to_numpy(),
                "spatial_lag": test["neighbour_prev_month_scaled"].to_numpy(),
                "log_offset": test["log_offset"].to_numpy(),
            },
            coords={"obs_id": np.arange(len(test))},
        )

        ppc_test = pm.sample_posterior_predictive(
            idata,
            var_names=["y_obs"],
            random_seed=random_seed,
            return_inferencedata=False,
            progressbar=True,
        )

    y_pred_samples = ppc_test["y_obs"]
    if y_pred_samples.ndim == 3:
        y_pred_samples = y_pred_samples.reshape(-1, y_pred_samples.shape[-1])
    elif y_pred_samples.ndim != 2:
        raise ValueError(f"Unexpected prediction shape: {y_pred_samples.shape}")

    predictions = test.copy()
    predictions["predicted_mean"] = y_pred_samples.mean(axis=0)
    predictions["predicted_lower"] = np.quantile(y_pred_samples, 0.05, axis=0)
    predictions["predicted_upper"] = np.quantile(y_pred_samples, 0.95, axis=0)

    params_summary = az.summary(
        idata,
        var_names=["intercept", "sigma_lsoa", "sigma_month", "beta_time", "beta_spatial_lag", "alpha"],
    )

    diagnostics = {
        "max_r_hat": float(params_summary["r_hat"].max()),
        "min_ess_bulk": float(params_summary["ess_bulk"].min()),
        "n_divergences": int(idata.sample_stats["diverging"].sum().values),
    }

    return predictions, idata, params_summary, diagnostics


## 10. Evaluation helpers

In [ ]:
def evaluate_predictions(predictions: pd.DataFrame) -> dict:
    actual = predictions["crime_count"].to_numpy()
    predicted = predictions["predicted_mean"].to_numpy()

    mae = mean_absolute_error(actual, predicted)
    rmse = mean_squared_error(actual, predicted) ** 0.5

    total_actual = actual.sum()
    total_predicted = predicted.sum()
    total_error = total_actual - total_predicted
    total_pct_error = np.nan if total_actual == 0 else total_error / total_actual

    coverage_90 = (
        (predictions["crime_count"] >= predictions["predicted_lower"])
        & (predictions["crime_count"] <= predictions["predicted_upper"])
    ).mean()

    return {
        "test_total": float(total_actual),
        "predicted_total": float(total_predicted),
        "total_error": float(total_error),
        "total_pct_error": float(total_pct_error),
        "abs_total_pct_error": float(abs(total_pct_error)),
        "model_mae": float(mae),
        "model_rmse": float(rmse),
        "coverage_90": float(coverage_90),
    }


def evaluate_lsoa_mean_baseline(test: pd.DataFrame, train: pd.DataFrame) -> dict:
    actual = test["crime_count"].to_numpy()

    baseline = (
        train.groupby("lsoa_code")["crime_count"]
        .mean()
        .rename("baseline_predicted_count")
    )
    eval_df = test.merge(baseline, on="lsoa_code", how="left")
    eval_df["baseline_predicted_count"] = eval_df["baseline_predicted_count"].fillna(train["crime_count"].mean())

    pred = eval_df["baseline_predicted_count"].to_numpy()
    mae = mean_absolute_error(actual, pred)
    rmse = mean_squared_error(actual, pred) ** 0.5

    total_actual = actual.sum()
    total_predicted = pred.sum()
    total_error = total_actual - total_predicted
    total_pct_error = np.nan if total_actual == 0 else total_error / total_actual

    return {
        "baseline_predicted_total": float(total_predicted),
        "baseline_total_error": float(total_error),
        "baseline_total_pct_error": float(total_pct_error),
        "baseline_abs_total_pct_error": float(abs(total_pct_error)),
        "baseline_mae": float(mae),
        "baseline_rmse": float(rmse),
    }


def extract_beta_spatial_lag(summary: pd.DataFrame) -> dict:
    if "beta_spatial_lag" not in summary.index:
        return {
            "beta_spatial_lag_mean": np.nan,
            "beta_spatial_lag_hdi_low": np.nan,
            "beta_spatial_lag_hdi_high": np.nan,
        }

    row = summary.loc["beta_spatial_lag"]
    hdi_low_col = "hdi_3%" if "hdi_3%" in summary.columns else "hdi_3"
    hdi_high_col = "hdi_97%" if "hdi_97%" in summary.columns else "hdi_97"

    return {
        "beta_spatial_lag_mean": float(row.get("mean", np.nan)),
        "beta_spatial_lag_hdi_low": float(row.get(hdi_low_col, np.nan)),
        "beta_spatial_lag_hdi_high": float(row.get(hdi_high_col, np.nan)),
    }


def build_prediction_output(predictions: pd.DataFrame, train: pd.DataFrame, experiment_id: str) -> pd.DataFrame:
    baseline = train.groupby("lsoa_code")["crime_count"].mean().rename("baseline_predicted_count")
    out = predictions.merge(baseline, on="lsoa_code", how="left")
    out["baseline_predicted_count"] = out["baseline_predicted_count"].fillna(train["crime_count"].mean())
    out["experiment_id"] = experiment_id

    out = out.rename(
        columns={
            "crime_count": "observed_count",
            "predicted_mean": "predicted_count",
            "predicted_lower": "lower_ci",
            "predicted_upper": "upper_ci",
        }
    )

    keep_cols = [
        "experiment_id",
        "police_force",
        "crime_type",
        "lsoa_code",
        "lsoa_name",
        "month",
        "observed_count",
        "predicted_count",
        "lower_ci",
        "upper_ci",
        "baseline_predicted_count",
        "neighbour_prev_month_count",
        "neighbour_prev_month_scaled",
        "prev_month_count",
    ]
    return out[[c for c in keep_cols if c in out.columns]].copy()



def add_derived_metric_columns(df: pd.DataFrame) -> pd.DataFrame:
    # add the alias and fallback metric columns used by the summary and detail tables
    df = df.copy()

    aliases = {
        "mae_improvement_vs_baseline": "mae_improvement",
        "rmae_vs_baseline": "rmae",
        "rmse_improvement_vs_baseline": "rmse_improvement",
        "runtime_seconds": "runtime_s",
    }
    for source_col, alias_col in aliases.items():
        if source_col in df.columns and alias_col not in df.columns:
            df[alias_col] = df[source_col]

    has_mae = {"baseline_mae", "model_mae"}.issubset(df.columns)
    has_rmse = {"baseline_rmse", "model_rmse"}.issubset(df.columns)

    if "mae_improvement" not in df.columns and has_mae:
        df["mae_improvement"] = df["baseline_mae"] - df["model_mae"]
    if "rmae" not in df.columns and has_mae:
        df["rmae"] = df["model_mae"] / df["baseline_mae"]
    if "rmse_improvement" not in df.columns and has_rmse:
        df["rmse_improvement"] = df["baseline_rmse"] - df["model_rmse"]
    if "runtime_s" not in df.columns:
        df["runtime_s"] = np.nan
    if "abs_total_pct_error" not in df.columns:
        df["abs_total_pct_error"] = np.nan

    return df


## 11. Single force-crime experiment runner

In [ ]:
def run_one_force_crime_experiment(
    experiment: dict,
    police_force: str,
    crime_type: str,
) -> tuple[dict, pd.DataFrame | None]:
    start_time = time.time()
    experiment = resolve_experiment_windows(experiment)
    panel_start, panel_end = experiment_panel_bounds(experiment)

    try:
        panel, panel_metadata = build_lsoa_month_panel(
            parquet_path=CRIME_PARQUET_PATH,
            police_force=police_force,
            crime_type=crime_type,
            panel_start_month=panel_start,
            panel_end_month=panel_end,
            train_windows=experiment["train_windows"],
            lsoa_reference=lsoa_centroids,
            max_lsoas=MAX_LSOAS_PER_MODEL,
            min_total_crimes=MIN_TOTAL_CRIMES_PER_LSOA,
            min_centroid_coverage=MIN_CENTROID_COVERAGE,
        )

        panel["log_offset"] = 0.0

        panel_adj, adjacency, adjacency_summary = add_adjacency_features_to_panel(
            panel,
            lsoa_reference=lsoa_centroids,
            k_neighbours=ADJACENCY_K_NEIGHBOURS,
        )

        train, test = split_train_test_for_experiment(panel_adj, experiment)
        train, test, scale_info = scale_features_using_train(train, test)

        baseline_metrics = evaluate_lsoa_mean_baseline(test, train)

        print(
            f"Fitting {experiment['experiment_id']} | {police_force} | {crime_type} | "
            f"train={experiment['train_windows']} | test={experiment['test_window']}"
        )

        predictions, idata, params_summary, diagnostics = fit_bayesian_nb_model(
            train=train,
            test=test,
            draws=DRAWS,
            tune=TUNE,
            chains=CHAINS,
            target_accept=TARGET_ACCEPT,
            random_seed=RANDOM_SEED,
        )

        prediction_output = build_prediction_output(predictions, train, experiment["experiment_id"])
        beta_spatial = extract_beta_spatial_lag(params_summary)

        if experiment["is_operational_forecast"]:
            model_metrics = {
                "test_total": np.nan,
                "predicted_total": float(predictions["predicted_mean"].sum()),
                "total_error": np.nan,
                "total_pct_error": np.nan,
                "abs_total_pct_error": np.nan,
                "model_mae": np.nan,
                "model_rmse": np.nan,
                "coverage_90": np.nan,
            }
            baseline_metrics = {
                "baseline_predicted_total": float(prediction_output["baseline_predicted_count"].sum()),
                "baseline_total_error": np.nan,
                "baseline_total_pct_error": np.nan,
                "baseline_abs_total_pct_error": np.nan,
                "baseline_mae": np.nan,
                "baseline_rmse": np.nan,
            }
            winner_mae = "no ground truth yet"
        else:
            model_metrics = evaluate_predictions(predictions)
            winner_mae = (
                "Bayesian NB adjacency"
                if model_metrics["model_mae"] < baseline_metrics["baseline_mae"]
                else "LSOA mean baseline"
            )

        rmae_vs_baseline = (
            np.nan
            if pd.isna(model_metrics["model_mae"]) or baseline_metrics["baseline_mae"] == 0 or pd.isna(baseline_metrics["baseline_mae"])
            else model_metrics["model_mae"] / baseline_metrics["baseline_mae"]
        )

        result = {
            "experiment_id": experiment["experiment_id"],
            "description": experiment["description"],
            "police_force": police_force,
            "crime_type": crime_type,
            "model_status": "success",
            "train_windows": str(experiment["train_windows"]),
            "test_window": str(experiment["test_window"]),
            "is_operational_forecast": bool(experiment["is_operational_forecast"]),
            "n_lsoas": int(train["lsoa_code"].nunique()),
            "n_train_rows": int(len(train)),
            "n_test_rows": int(len(test)),
            "centroid_coverage": panel_metadata["centroid_coverage"],
            "n_lsoas_dropped_no_centroid": panel_metadata["n_lsoas_dropped_no_centroid"],
            "mean_neighbours": float(adjacency_summary.get("mean_neighbours", np.nan)),
            **baseline_metrics,
            **model_metrics,
            "mae_improvement_vs_baseline": (
                np.nan
                if pd.isna(model_metrics["model_mae"]) or pd.isna(baseline_metrics["baseline_mae"])
                else baseline_metrics["baseline_mae"] - model_metrics["model_mae"]
            ),
            "rmse_improvement_vs_baseline": (
                np.nan
                if pd.isna(model_metrics["model_rmse"]) or pd.isna(baseline_metrics["baseline_rmse"])
                else baseline_metrics["baseline_rmse"] - model_metrics["model_rmse"]
            ),
            "rmae_vs_baseline": rmae_vs_baseline,
            "winner_mae": winner_mae,
            **beta_spatial,
            "max_r_hat": diagnostics["max_r_hat"],
            "min_ess_bulk": diagnostics["min_ess_bulk"],
            "n_divergences": diagnostics["n_divergences"],
            "runtime_seconds": float(time.time() - start_time),
            "error": "",
        }

        return result, prediction_output

    except Exception as exc:
        return {
            "experiment_id": experiment["experiment_id"],
            "description": experiment["description"],
            "police_force": police_force,
            "crime_type": crime_type,
            "model_status": "failed",
            "train_windows": str(experiment.get("train_windows")),
            "test_window": str(experiment.get("test_window")),
            "is_operational_forecast": bool(experiment.get("is_operational_forecast", False)),
            "runtime_seconds": float(time.time() - start_time),
            "error": str(exc),
        }, None


## 12. Build the list of model runs

In [ ]:
selected_experiments = EXPERIMENTS if EXPERIMENTS_TO_RUN == "all" else [
    exp for exp in EXPERIMENTS if exp["experiment_id"] in EXPERIMENTS_TO_RUN
]

experiment_runs = []
for exp in selected_experiments:
    for force in SELECTED_POLICE_FORCES:
        for crime in SELECTED_CRIME_TYPES:
            experiment_runs.append(
                {
                    "experiment": exp,
                    "police_force": force,
                    "crime_type": crime,
                }
            )

print("Total model runs:", len(experiment_runs))
pd.DataFrame(
    [
        {
            "experiment_id": r["experiment"]["experiment_id"],
            "police_force": r["police_force"],
            "crime_type": r["crime_type"],
        }
        for r in experiment_runs
    ]
).head(20)


## 13. Run all experiments

In [ ]:
results = []
prediction_outputs = []

checkpoint_path = TABLES_DIR / f"{OUTPUT_TAG}_checkpoint.csv"

for i, run in enumerate(experiment_runs, start=1):
    print("=" * 120)
    print(f"Run {i}/{len(experiment_runs)}")
    print(run["experiment"]["experiment_id"], "|", run["police_force"], "|", run["crime_type"])
    print("=" * 120)

    result, preds = run_one_force_crime_experiment(
        experiment=run["experiment"],
        police_force=run["police_force"],
        crime_type=run["crime_type"],
    )

    results.append(result)
    pd.DataFrame(results).to_csv(checkpoint_path, index=False)

    if preds is not None:
        prediction_outputs.append(preds)

    display(pd.DataFrame([result]))

results_df = pd.DataFrame(results)
results_df


## 14. Results overview

In [ ]:
successful = results_df[results_df["model_status"] == "success"].copy()
failed = results_df[results_df["model_status"] != "success"].copy()

print("Successful runs:", len(successful))
print("Failed runs:", len(failed))

readable_cols = [
    "experiment_id",
    "police_force",
    "crime_type",
    "model_status",
    "n_lsoas",
    "baseline_mae",
    "model_mae",
    "mae_improvement_vs_baseline",
    "rmae_vs_baseline",
    "baseline_rmse",
    "model_rmse",
    "rmse_improvement_vs_baseline",
    "coverage_90",
    "winner_mae",
    "beta_spatial_lag_mean",
    "beta_spatial_lag_hdi_low",
    "beta_spatial_lag_hdi_high",
    "max_r_hat",
    "min_ess_bulk",
    "n_divergences",
    "runtime_seconds",
]

display(successful[[c for c in readable_cols if c in successful.columns]].sort_values(["experiment_id", "police_force", "crime_type"]))

if len(failed) > 0:
    display(failed[["experiment_id", "police_force", "crime_type", "error", "runtime_seconds"]])


## 15. Aggregate summaries by regime, force, and crime type

In [ ]:
evaluated = successful[~successful["is_operational_forecast"]].copy()
evaluated = add_derived_metric_columns(evaluated)

evaluated["diagnostic_flag"] = np.where(
    (evaluated["max_r_hat"] > 1.05) | (evaluated["min_ess_bulk"] < 50),
    "Use with caution",
    "OK",
)

if not evaluated.empty:
    experiment_summary = (
        evaluated
        .groupby("experiment_id", as_index=False)
        .agg(
            n_runs=("crime_type", "count"),
            mean_baseline_mae=("baseline_mae", "mean"),
            mean_model_mae=("model_mae", "mean"),
            mean_mae_improvement=("mae_improvement", "mean"),
            mean_rmae=("rmae", "mean"),
            model_win_rate=("winner_mae", lambda s: (s == "Bayesian NB adjacency").mean()),
            mean_baseline_rmse=("baseline_rmse", "mean"),
            mean_model_rmse=("model_rmse", "mean"),
            mean_rmse_improvement=("rmse_improvement", "mean"),
            mean_abs_total_pct_error=("abs_total_pct_error", "mean"),
            mean_coverage_90=("coverage_90", "mean"),
            total_runtime_minutes=("runtime_s", lambda s: s.sum() / 60),
            max_r_hat=("max_r_hat", "max"),
            min_ess_bulk=("min_ess_bulk", "min"),
            total_divergences=("n_divergences", "sum"),
            caution_share=("diagnostic_flag", lambda s: (s == "Use with caution").mean()),
        )
    )

    experiment_summary["mean_mae_improvement_pct"] = (
        experiment_summary["mean_mae_improvement"]
        / experiment_summary["mean_baseline_mae"]
    )

    display(experiment_summary)

    force_summary = (
        evaluated
        .groupby(["experiment_id", "police_force"], as_index=False)
        .agg(
            n_runs=("crime_type", "count"),
            mean_baseline_mae=("baseline_mae", "mean"),
            mean_model_mae=("model_mae", "mean"),
            mean_mae_improvement=("mae_improvement", "mean"),
            mean_rmae=("rmae", "mean"),
            model_win_rate=("winner_mae", lambda s: (s == "Bayesian NB adjacency").mean()),
            mean_abs_total_pct_error=("abs_total_pct_error", "mean"),
            mean_coverage_90=("coverage_90", "mean"),
            max_r_hat=("max_r_hat", "max"),
            min_ess_bulk=("min_ess_bulk", "min"),
            total_divergences=("n_divergences", "sum"),
            caution_share=("diagnostic_flag", lambda s: (s == "Use with caution").mean()),
        )
        .sort_values(["experiment_id", "mean_mae_improvement"], ascending=[True, False])
    )

    display(force_summary)

    crime_summary = (
        evaluated
        .groupby(["experiment_id", "crime_type"], as_index=False)
        .agg(
            n_runs=("police_force", "count"),
            mean_baseline_mae=("baseline_mae", "mean"),
            mean_model_mae=("model_mae", "mean"),
            mean_mae_improvement=("mae_improvement", "mean"),
            mean_rmae=("rmae", "mean"),
            model_win_rate=("winner_mae", lambda s: (s == "Bayesian NB adjacency").mean()),
            mean_abs_total_pct_error=("abs_total_pct_error", "mean"),
            mean_coverage_90=("coverage_90", "mean"),
            max_r_hat=("max_r_hat", "max"),
            min_ess_bulk=("min_ess_bulk", "min"),
            total_divergences=("n_divergences", "sum"),
            caution_share=("diagnostic_flag", lambda s: (s == "Use with caution").mean()),
        )
        .sort_values(["experiment_id", "mean_mae_improvement"], ascending=[True, False])
    )

    display(crime_summary)

else:
    print("No evaluated experiments available yet.")

## 16. Summary charts

In [ ]:
if not evaluated.empty:
    plot_df = experiment_summary.sort_values("experiment_id")

    plt.figure(figsize=(10, 5))
    plt.bar(plot_df["experiment_id"], plot_df["mean_mae_improvement"])
    plt.axhline(0, linestyle="--")
    plt.ylabel("Mean MAE improvement vs baseline")
    plt.title("Model improvement by train/test regime")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.bar(plot_df["experiment_id"], plot_df["model_win_rate"])
    plt.ylim(0, 1)
    plt.ylabel("Share of force-crime runs where model beats baseline")
    plt.title("Model win rate by train/test regime")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, 5))
    plt.bar(plot_df["experiment_id"], plot_df["mean_abs_total_pct_error"])
    plt.ylabel("Mean absolute total prediction error")
    plt.title("Aggregate calibration error by regime")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()


## 17. Per-run detail table

In [ ]:
if not evaluated.empty:
    compact = add_derived_metric_columns(evaluated)

    display_cols = [
        "experiment_id",
        "police_force",
        "crime_type",
        "baseline_mae",
        "model_mae",
        "mae_improvement",
        "rmae",
        "baseline_rmse",
        "model_rmse",
        "rmse_improvement",
        "abs_total_pct_error",
        "coverage_90",
        "winner_mae",
        "max_r_hat",
        "min_ess_bulk",
        "n_divergences",
    ]

    display_cols = [c for c in display_cols if c in compact.columns]

    compact = compact[display_cols].copy()

    round_cols = [
        "baseline_mae",
        "model_mae",
        "mae_improvement",
        "rmae",
        "baseline_rmse",
        "model_rmse",
        "rmse_improvement",
        "abs_total_pct_error",
        "coverage_90",
        "max_r_hat",
    ]

    for col in round_cols:
        if col in compact.columns:
            compact[col] = compact[col].round(3)

    sort_cols = [c for c in ["experiment_id", "police_force", "crime_type"] if c in compact.columns]

    display(compact.sort_values(sort_cols))
else:
    print("No evaluated experiments available yet.")

## 18. Save outputs

In [ ]:
if SAVE_RESULTS:
    results_path = TABLES_DIR / f"{OUTPUT_TAG}_metrics.csv"
    results_df.to_csv(results_path, index=False)
    print("Saved metrics:", results_path)

    if prediction_outputs:
        predictions_all = pd.concat(prediction_outputs, ignore_index=True)
        predictions_path = TABLES_DIR / f"{OUTPUT_TAG}_predictions.csv"
        predictions_all.to_csv(predictions_path, index=False)
        print("Saved predictions:", predictions_path)
    else:
        predictions_all = pd.DataFrame()
else:
    predictions_all = pd.concat(prediction_outputs, ignore_index=True) if prediction_outputs else pd.DataFrame()

print("Prediction rows:", len(predictions_all))
if not predictions_all.empty:
    display(predictions_all.head(10))
